# Gallstone Final Report
- **Group 18**
    - Timothy
    - Elmeri
    - Pony
    - Miguel
- **STAT 301**
- **Dec 6 2025**

## Introduction
#### Summary
- The dataset assigned to our group is about Gallstone Disease by Esen, Irfan, et al. It contains bioimpedance and laboratory data to develop machine learning models for predicting gallstone disease, which was served as an alternative diagnostic approach to traditional imaging techniques such as ultrasonography, CT, and MRI (Esen, Irfan, et al.). Specifically, the dataset contains 319 observations of which 161 are gallstone patients and 158 are healthy controls (Esen, Irfan, et al.). Furthermore, the dataset includes 38 variables with no missing values (Esen, Irfan, et al.), and their name, type, and description are given in the table below:

#### Research Focus
- The primary focus of our question will be inference. We want to understand and quantify the relationship between each predictor and the odds of having gallstones. Specifically, we aim to determine which predictors are statistically significantly associated with gallstone disease.

#### Research Question
- Which bioimpedance measurements and laboratory biomarkers are significantly associated with gallstone disease?

#### Source & Information
- This dataset originates from a prospective descriptive study conducted at the Internal Medicine Outpatient Clinic of Ankara VM Medical Park Hospital between June 2022 and June 2023 (Esen, Irfan, et al.) From an initial 454 participants, 134 individuals who had undergone gallbladder surgery were excluded, resulting in the final dataset of 319 observations of which 161 are gallstone patients and 158 are healthy controls (Esen, Irfan, et al.).
- **Citation**
    - Esen, Irfan, et al. "Gallstone." UCI Machine Learning Repository, 2024, https://doi.org/10.1097/md.0000000000037258.

## Methods and Results

### Data

In [1]:
# load required libraries
library(readxl)
library(tidyverse)
library(ggplot2)
library(reshape2)
library(patchwork)
library(broom)

# set plot size
options(repr.plot.width = 15, repr.plot.height = 12)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.5.2
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths




In [5]:
# Download the zip file containing the dataset
outer_zip <- tempfile(fileext = ".zip")
download.file(
  "https://archive.ics.uci.edu/static/public/1150/gallstone-1.zip",
  outer_zip,
  mode = "wb"
)

# Unzip the outer zip into tempdir
tmp <- tempdir()
unzip(outer_zip, exdir = tmp)

# Locate the inner ZIP (dataset-uci.zip)
inner_zip <- list.files(
  tmp,
  pattern = "dataset-uci\\.zip$",
  full.names = TRUE,
  recursive = TRUE
)
inner_zip <- inner_zip[1]

# Unzip the inner zip file
unzip(inner_zip, exdir = tmp)

# Locate the xlsx data file
xlsx_path <- list.files(
  tmp,
  pattern = "dataset-uci\\.xlsx$",
  full.names = TRUE,
  recursive = TRUE
)

xlsx_path <- xlsx_path[1]

# Read the Excel file
gallstone_data <- read_xlsx(xlsx_path)

head(gallstone_data)

Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,⋯,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,50,0,0,0,0,0,0,185,92.8,⋯,40,134,20,22,87,0.82,112.47,0,16.0,33.0
0,47,0,1,0,0,0,0,176,94.5,⋯,43,103,14,13,46,0.87,107.10,0,14.4,25.0
0,61,0,0,0,0,0,0,171,91.1,⋯,43,69,18,14,66,1.25,65.51,0,16.2,30.2
0,41,0,0,0,0,0,0,168,67.7,⋯,59,53,20,12,34,1.02,94.10,0,15.4,35.4
0,42,0,0,0,0,0,0,178,89.6,⋯,30,326,27,54,71,0.82,112.47,0,16.8,40.6
0,96,0,0,0,0,0,0,155,49.0,⋯,30,65,13,13,60,1.46,43.74,0,11.0,45.8


### About the data

The dataset contains clinical data collected from the Internal Medicine Outpatient Clinic of Ankara VM Medical Park Hospital. The data represent 319 individuals with 116 individuals diagnosed with gallstone disease. There are 38 features representing demographic, biological, and laboratory data.

Citation:

Esen, I., Arslan, H., Aktürk, S., Gülşen, M., Kültekin, N., & Özdemir, O. (2024). Gallstone [Dataset]. UCI Machine Learning Repository. https://doi.org/10.1097/md.0000000000037258.

| Variable Name | Type | Description | Number of Observations |
|---------------|------|-------------|------------------------|
| Gallstone Status | Binary | Gallstones present (0) or absent (1) | 319 |
| Age | Integer | Age of the person | 319 |
| Gender | Categorical | Gender of the person | 319 |
| Comorbidity | Categorical | Other diseases that the person has: 0 (No comorbidities present), 1 (One comorbid condition), 2 (Two comorbid conditions), 3 (Three or more comorbid conditions)| 319 |
| Coronary Artery Disease (CAD) | Binary | Cardiovascular disease: 0 (No), 1 (Yes)| 319 |
| Hypothyroidism | Binary | Underactive thyroid gland: 0 (No), 1 (Yes) | 319 |
| Hyperlipidemia | Binary | High levels of fat in the blood: 0 (No), 1 (Yes) | 319 |
| Diabetes Mellitus (DM) | Binary | High blood sugar: 0 (No), 1 (Yes) | 319 |
| Height | Integer | Height in cm | 319|
| Weight | Numeric | Weight in kg | 319|
| Body Mass Index (BMI) | Numeric | Weight to height ratio | 319 |
| Total Body Water (TBW) | Numeric | Total water in the body | 319 |
| Extracellular Water (ECW) | Numeric | Extracellular water | 319 |
| Intracellular Water (ICW) | Numeric | Intracellular water | 319 |
| Extracellular Fluid/Total Body Water (ECF/TBW) | Numeric | Extracellular water content | 319 |
| Total Body Fat Ratio (TBFR) (%) | Numeric | Total fat content (%) | 319 |
| Lean Mass (LM) (%) | Numeric | Lean body mass (%) | 319 |
| Body Protein Content (Protein) (%) | Numeric | Body protein content (%) | 319 |
| Visceral Fat Rating (VFR) | Integer | Visceral organ fat level | 319 |
| Bone Mass (BM) | Numeric | Mass of bones | 319 |
| Muscle Mass (MM) | Numeric | Muscle mass | 319 |
| Obesity (%) | Numeric | Excess fat (%) | 319 |
| Total Fat Content (TFC) | Numeric | Total fat amount | 319 |
| Visceral Fat Area (VFA) | Numeric | Inner adipose tissue area | 319 |
| Visceral Muscle Area (VMA) (Kg) | Numeric | Inner muscle area (kg) | 319 |
| Hepatic Fat Accumulation (HFA) | Categorical | Accumulation of fat in the liver: 0 (No fat accumulation),1 (Grade 1 (mild)), 2 (Grade 2 (moderate)), 3 (Grade 3 (severe)), 4 (Grade 4 (very severe)) | 319 |
| Glucose | Numeric | Blood sugar | 319 |
| Total Cholesterol (TC) | Numeric | Total cholesterol amount | 319 |
| Low Density Lipoprotein (LDL) | Numeric | Amount of "bad" cholesterol | 319 |
| High Density Lipoprotein (HDL) | Numeric | Amount of "good" cholesterol | 319 |
| Triglyceride | Numeric | Amount of trigliceride fats found in blood | 319 |
| Aspartate Aminotransferase (AST) | Numeric | Amount of liver enzyme | 319 |
| Alanine Aminotransferase (ALT) | Numeric | Amount of liver enzyme | 319 |
| Alkaline Phosphatase (ALP) | Numeric | Amount of liver and bone enzyme | 319 |
| Creatinine | Numeric | Kidney function indicator | 319 |
| Glomerular Filtration Rate (GFR) | Numeric | Kidney filtration rate | 319 |
| C-Reactive Protein (CRP) | Numeric | Inflammation indicator | 319 |
| Hemoglobin (HGB) | Numeric | Protein that carries oxygen in the blood | 319 |
| Vitamin D | Numeric | Essential vitamin for bone health | 319 |

### Exploratory Data Analysis